# 01_EDA_limpieza.ipynb
Exploración y limpieza de datos del dataset de precios de propiedades.


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
from preprocessing import parse_amount, parse_area, parse_floor, parse_parking, extract_bhk
sns.set(style='whitegrid', font_scale=1.1)
df = pd.read_csv(os.path.abspath(os.path.join('..', 'data', 'raw', 'house_prices.csv')))
print('Shape:', df.shape)
print(df.dtypes)
print((df.isnull().mean() * 100).round(2).sort_values(ascending=False))
df = df.drop(columns=['Dimensions', 'Plot Area'], errors='ignore')
df.head()


## Parseo de columnas sucias
Convertimos las columnas de texto en variables numéricas y extraemos información útil.


In [ ]:
df['amount_rupees'] = df['Amount(in rupees)'].apply(parse_amount)
df['carpet_area_num'] = df['Carpet Area'].apply(parse_area)
df['super_area_num'] = df['Super Area'].apply(parse_area)
df[['floor_number', 'total_floors']] = df['Floor'].apply(lambda x: pd.Series(parse_floor(x)))
df[['has_parking', 'parking_type']] = df['Car Parking'].apply(lambda x: pd.Series(parse_parking(x)))
df['bhk_count'] = df['Title'].apply(extract_bhk)
df[['amount_rupees', 'carpet_area_num', 'super_area_num', 'floor_number', 'total_floors', 'has_parking', 'bhk_count']].head()


## Tratamiento de datos faltantes
Aplico estrategias de imputación según la naturaleza de cada variable.


In [ ]:
df['Balcony'] = df['Balcony'].astype(float)
df['Bathroom'] = df['Bathroom'].astype(float)
df['bhk_count'] = df['bhk_count'].fillna(0).astype(int)
balcony_modes = df.groupby('bhk_count')['Balcony'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else 0)
df['Balcony'] = df.apply(\n
    lambda row: balcony_modes.loc[row['bhk_count']] if pd.isna(row['Balcony']) else row['Balcony'],\n
    axis=1
 )
df['Bathroom'] = df['Bathroom'].fillna(df['Bathroom'].median())
df['carpet_area_num'] = df.groupby('bhk_count')['carpet_area_num'].transform(lambda x: x.fillna(x.median()))
df['super_area_num'] = df.groupby('bhk_count')['super_area_num'].transform(lambda x: x.fillna(x.median()))
df['facing'] = df['facing'].fillna('Unknown')
df['overlooking'] = df['overlooking'].fillna('Unknown')
df['Ownership'] = df['Ownership'].fillna(df['Ownership'].mode().iloc[0])
df['Status'] = df['Status'].fillna(df['Status'].mode().iloc[0])
df['has_society'] = df['Society'].notna().astype(int)
df[['Balcony', 'Bathroom', 'carpet_area_num', 'super_area_num', 'facing', 'overlooking', 'Ownership', 'Status', 'has_society']].isnull().mean().round(4) * 100


## Detección de outliers con IQR
Identificamos outliers en precio y área antes de continuar.


In [ ]:
from preprocessing import remove_outliers_iqr
for col in ['amount_rupees', 'carpet_area_num']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    print(f'{col}: lower={lower:.2f}, upper={upper:.2f}')
before_count = df.shape[0]
df = remove_outliers_iqr(df, 'amount_rupees')
df = remove_outliers_iqr(df, 'carpet_area_num')
after_count = df.shape[0]
print(f'Registros eliminados por outliers: {before_count - after_count}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x=df['amount_rupees'], ax=axes[0], color='skyblue')
axes[0].set_title('Precio Total (amount_rupees)')
sns.boxplot(x=df['carpet_area_num'], ax=axes[1], color='lightgreen')
axes[1].set_title('Área útil (carpet_area_num)')
plt.tight_layout()
plt.savefig(os.path.abspath(os.path.join('..', 'reports', 'figures', '01_boxplots_eda.png')), dpi=150, bbox_inches='tight')
plt.show()


## Guardar dataset limpio
Guardo el resultado de la limpieza para usarlo en las siguientes fases.


In [ ]:
df.to_csv(os.path.abspath(os.path.join('..', 'data', 'processed', 'house_prices_clean.csv')), index=False)
print('Datos limpios guardados en data/processed/house_prices_clean.csv')
